# Aprendizado de Máquina — Aula prática 10

## KNN e Árvores de Classificação

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

As Aulas 04 e 05 apresentaram dois métodos não paramétricos de regressão, o KNN e as
árvores. Esta aula os adapta a uma resposta qualitativa, e quase toda a adaptação está em
duas trocas: o KNN e as folhas de uma árvore passam a resumir as respostas de treino pela
moda, a classe mais frequente, em vez da média; e as árvores passam a escolher as divisões
por uma medida de impureza das classes, em vez da soma de quadrados.

Este notebook acompanha as notas da aula, na mesma ordem e com a mesma notação. A Seção 2
estuda o KNN nos dados da Figura 1 das notas, uma população em que o classificador de Bayes
é conhecido. A Seção 3 compara as três medidas de impureza dos slides e mostra por que o erro
de classificação não serve para crescer a árvore. A Seção 4 cresce e poda uma árvore no
exemplo dos slides e confere que medida o `scikit-learn` usa em cada etapa. A Seção 5 agrega
árvores de classificação em *bagging* e em florestas aleatórias.

### Objetivos

Ao final deste notebook, você deve ser capaz de:

- reconhecer o KNN de classificação como o voto dos $k$ vizinhos mais próximos e o
  `predict_proba` como a proporção de vizinhos de cada classe, e ver como essa proporção varia
  de uma amostra de treino para outra;
- descrever o classificador nos extremos $k = 1$ e $k = n$, escolher $k$ por validação cruzada
  e comparar o resultado com o classificador de Bayes;
- ver como covariáveis irrelevantes prejudicam o KNN;
- calcular o erro de classificação, o índice de Gini e a entropia de uma região e a impureza
  ponderada de uma divisão, e verificar que o erro de classificação não registra divisões que
  aumentam a pureza;
- crescer e podar uma árvore de classificação no `scikit-learn`, sabendo que medida ele usa em
  cada etapa, e comparar o Gini com a entropia;
- agregar árvores de classificação em *bagging* e em florestas aleatórias, pelo voto ou pela
  média das probabilidades.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

Os estimadores novos do `scikit-learn` são as versões de classificação dos que usamos em
regressão nas Aulas 04 e 05: o `KNeighborsClassifier`, o `DecisionTreeClassifier`, o
`BaggingClassifier` e o `RandomForestClassifier`. O `make_moons` gera os dados do exemplo dos
slides, e o `norm`, do `scipy`, dá a função de distribuição da normal padrão, que a Seção 2 usa
para calcular a acurácia do classificador de Bayes.

In [ ]:
import sklearn.model_selection as skm
from scipy.stats import norm
from sklearn.datasets import make_moons
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [ ]:
import warnings
warnings.filterwarnings("ignore")

A função `desenhar`, a seguir, faz as figuras do notebook. Ela calcula a classe que o modelo
atribui a cada ponto de uma grade fina do plano e pinta o fundo com a cor dessa classe, azul
para a classe 0 e vermelho para a classe 1; por cima, desenha as observações, com as mesmas
cores. Sem modelo (`modelo=None`), desenha só as observações.

In [ ]:
def desenhar(ax, modelo, X, y, titulo=""):
    if modelo is not None:
        x0 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300)
        x1 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
        gx, gy = np.meshgrid(x0, x1)
        classe = modelo.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
        ax.contourf(gx, gy, classe, levels=[-0.5, 0.5, 1.5],
                    colors=["steelblue", "crimson"], alpha=0.15)
    for c, cor in [(0, "steelblue"), (1, "crimson")]:
        ax.scatter(X[y == c, 0], X[y == c, 1], s=10, color=cor, edgecolor="k", lw=0.2)
    ax.set_title(titulo, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

---
## 2. O KNN em classificação

Em regressão, o KNN da Aula 04 estimava $E[Y \mid \mathbf{X} = \mathbf{x}]$ pela média das
respostas dos $k$ vizinhos mais próximos de $\mathbf{x}$. Em classificação, $Y$ assume valores
num conjunto finito de classes $\mathcal{C}$, e a média das respostas deixa de fazer sentido.
Como nas notas, sejam $d^k_{\mathbf{x}}$ a distância de $\mathbf{x}$ à $k$-ésima observação de
treino mais próxima e
$\mathcal{N}^{(k)}_{\mathbf{x}} = \{i : d(\mathbf{x}_i, \mathbf{x}) \le d^k_{\mathbf{x}}\}$ o
conjunto dos índices das $k$ observações de treino mais próximas de $\mathbf{x}$. O KNN de
classificação atribui a $\mathbf{x}$ a classe mais frequente entre os vizinhos,

$$\widehat g(\mathbf{x}) = \operatorname{moda}\{y_i : i \in \mathcal{N}^{(k)}_{\mathbf{x}}\}:$$

cada vizinho vota na sua classe, e vence a mais votada. Nada nessa definição depende do número
de classes, e o mesmo método serve a problemas com mais de duas.

Os dados desta seção são os da Figura 1 das notas, gerados com a mesma semente. São duas
classes, cada uma com distribuição normal bivariada de coordenadas independentes e
desvio-padrão $\sigma = 0{,}95$, centradas em $\boldsymbol{\mu}_0 = (-0{,}9;\,-0{,}4)$, a
classe 0, e em $\boldsymbol{\mu}_1 = (1{,}1;\,0{,}8)$, a classe 1. O treino tem 150 observações
de cada classe, e o desempenho em observações novas é medido em outras 10 000, 5 000 de cada
classe.

Nessa população, o classificador de Bayes da Aula 07 é conhecido. As duas classes têm a mesma
probabilidade a priori e densidades normais com a mesma matriz de covariância $\sigma^2 I$;
então a classe mais provável em $\mathbf{x}$ é a de maior densidade em $\mathbf{x}$, e a
densidade de cada classe decresce com a distância de $\mathbf{x}$ ao seu centro. O
classificador de Bayes atribui $\mathbf{x}$ à classe do centro mais próximo, e a sua
fronteira é a mediatriz do segmento entre $\boldsymbol{\mu}_0$ e $\boldsymbol{\mu}_1$, uma
reta. Para calcular a acurácia dele, seja
$\mathbf{u} = (\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0)/\|\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0\|$.
Uma observação $\mathbf{X}$ da classe 0 fica mais perto de $\boldsymbol{\mu}_1$ quando a
projeção $\mathbf{u}^\top(\mathbf{X} - \boldsymbol{\mu}_0)$ passa de
$\|\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0\|/2$, e essa projeção é normal com média 0 e
desvio-padrão $\sigma$, porque $\|\mathbf{u}\| = 1$. A probabilidade de erro na classe 0 é,
portanto, $1 - \Phi\big(\|\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0\|/(2\sigma)\big)$, em que
$\Phi$ é a função de distribuição da normal padrão; pela simetria, a da classe 1 é a mesma, e
a acurácia do classificador de Bayes é
$\Phi\big(\|\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0\|/(2\sigma)\big)$. Nenhum classificador
acerta, em média, mais que isso. A célula gera os dados, desenha o treino e calcula essa
acurácia, pela fórmula e nas 10 000 observações novas.

In [ ]:
# a populacao da Figura 1 das notas, com a mesma semente e os mesmos parametros
rng = np.random.default_rng(61)
n = 300
mu0, mu1, sigma = np.array([-0.9, -0.4]), np.array([1.1, 0.8]), 0.95
X_tr = np.vstack([rng.normal(mu0, sigma, size=(n // 2, 2)),
                  rng.normal(mu1, sigma, size=(n // 2, 2))])
y_tr = np.r_[np.zeros(n // 2), np.ones(n // 2)].astype(int)
X_te = np.vstack([rng.normal(mu0, sigma, size=(5000, 2)),
                  rng.normal(mu1, sigma, size=(5000, 2))])
y_te = np.r_[np.zeros(5000), np.ones(5000)].astype(int)

fig, ax = subplots(figsize=(4.6, 3.8))
desenhar(ax, None, X_tr, y_tr, "as 300 observacoes de treino")

distancia = np.linalg.norm(mu1 - mu0)
mais_perto_de_mu1 = np.linalg.norm(X_te - mu1, axis=1) < np.linalg.norm(X_te - mu0, axis=1)
acerto_bayes = np.mean(mais_perto_de_mu1.astype(int) == y_te)
print(f"||mu1 - mu0|| = {distancia:.4f};  ||mu1 - mu0|| / (2 sigma) = {distancia / (2 * sigma):.4f}")
print(f"acuracia do classificador de Bayes: {norm.cdf(distancia / (2 * sigma)):.4f} (formula), "
      f"{acerto_bayes:.4f} (nas 10000 observacoes novas)")

As duas nuvens se sobrepõem, e nenhuma fronteira separa as classes sem erro. Com
$\|\boldsymbol{\mu}_1 - \boldsymbol{\mu}_0\| = \sqrt{5{,}44} \approx 2{,}332$, a acurácia do
classificador de Bayes é $\Phi(1{,}228) \approx 0{,}890$. Nas 10 000 observações novas ele
acerta 0,892; a diferença, de dois milésimos, é a variação de uma amostra finita. Esse é o teto
com que vamos comparar o KNN.

O KNN é um classificador *plug-in* (Aula 07): para cada classe $c$, estima
$P(Y = c \mid \mathbf{X} = \mathbf{x})$ pela proporção de vizinhos dessa classe,

$$\widehat P(Y = c \mid \mathbf{x}) = \frac{1}{k} \sum_{i \in \mathcal{N}^{(k)}_{\mathbf{x}}} \mathbf{1}\{y_i = c\},$$

e, como $k\,\widehat P(Y = c \mid \mathbf{x})$ é o número de votos da classe $c$, a classe de
maior proporção é a mais votada. No `KNeighborsClassifier`, o `predict` devolve a classe mais
votada; o `predict_proba`, as proporções, uma coluna para cada classe, na ordem de `classes_`;
e o `kneighbors`, as distâncias e os índices dos $k$ vizinhos. A célula confere essas três
saídas, com $k = 15$, num ponto escolhido sem olhar os dados: o ponto médio entre os dois
centros, $\mathbf{x}_0 = (\boldsymbol{\mu}_0 + \boldsymbol{\mu}_1)/2 = (0{,}1;\,0{,}2)$. Ele está
sobre a fronteira de Bayes, onde as duas classes são igualmente prováveis:
$P(Y = 1 \mid \mathbf{X} = \mathbf{x}_0) = 1/2$.

In [ ]:
x0 = ((mu0 + mu1) / 2).reshape(1, -1)     # o ponto medio entre os dois centros
modelo = KNeighborsClassifier(n_neighbors=15).fit(X_tr, y_tr)
distancias, vizinhos = modelo.kneighbors(x0)
rotulos = y_tr[vizinhos[0]]

print(f"x0 = {x0[0]}")
print(f"classes dos 15 vizinhos de x0: {rotulos}")
print(f"votos: {np.sum(rotulos == 0)} na classe 0 e {np.sum(rotulos == 1)} na classe 1")
print(f"proporcao da classe 1 entre os vizinhos: {rotulos.mean():.4f}")
print(f"predict_proba(x0) = {modelo.predict_proba(x0)[0]}   (colunas na ordem de classes_ = {modelo.classes_})")
print(f"predict(x0) = {modelo.predict(x0)[0]}")

Dos 15 vizinhos de $\mathbf{x}_0$, 3 são da classe 1 e 12 da classe 0. A proporção da classe 1
é $3/15 = 0{,}2$, o número que o `predict_proba` devolve na segunda coluna, e a classe prevista
é a 0, a mais votada. A estimativa está longe da probabilidade verdadeira, $1/2$: ela depende
de quais observações de treino caíram perto de $\mathbf{x}_0$ nesta amostra, e com outra
amostra seria outra. A próxima célula mede essa dependência.

A célula a seguir sorteia mil amostras de treino da mesma população, cada uma com 150
observações de cada classe, e calcula em cada uma a proporção da classe 1 entre os $k$ vizinhos
de $\mathbf{x}_0$, para $k = 1$, $15$ e $120$, os três valores da Figura 1 das notas.

Antes, o que esperar da média. A reflexão em torno da fronteira de Bayes leva a distribuição da
classe 0 na da classe 1, e a da classe 1 na da classe 0, e deixa $\mathbf{x}_0$ no lugar,
preservando as distâncias a ele. Refletir uma amostra de treino e trocar as classes produz,
então, uma amostra com a mesma distribuição da original, em que os vizinhos de $\mathbf{x}_0$
são os reflexos dos vizinhos originais, com as classes trocadas. Logo, o número de vizinhos da
classe 1 tem a mesma
distribuição que o número de vizinhos da classe 0, e, em $\mathbf{x}_0$, a proporção da classe
1 tem esperança exatamente $1/2$, qualquer que seja $k$: ali o KNN não erra de forma
sistemática, e o que muda com $k$ é só a dispersão das estimativas.

In [ ]:
rng_rep = np.random.default_rng(1)
estimativas = {1: [], 15: [], 120: []}
for _ in range(1000):
    X_rep = np.vstack([rng_rep.normal(mu0, sigma, size=(n // 2, 2)),
                       rng_rep.normal(mu1, sigma, size=(n // 2, 2))])
    for k in estimativas:
        _, viz = KNeighborsClassifier(n_neighbors=k).fit(X_rep, y_tr).kneighbors(x0)
        estimativas[k].append(y_tr[viz[0]].mean())

for k, est in estimativas.items():
    est = np.array(est)
    print(f"k = {k:3d}: media das 1000 estimativas {est.mean():.4f}, desvio-padrao {est.std():.4f}")

A média das mil estimativas fica perto de $1/2$ nos três casos, como a simetria exige. O
desvio-padrão cai de 0,50 com $k = 1$, em que a estimativa só pode valer 0 ou 1, para 0,125
com $k = 15$ e 0,036 com $k = 120$. É a variância de que falam os slides: com $k$ pequeno, o
classificador depende de poucas observações e muda muito de uma amostra de treino para outra;
com $k$ grande, depende de muitas e muda pouco. Em $\mathbf{x}_0$, um $k$ grande não tem custo,
porque ali não há viés. Num ponto longe da fronteira, uma vizinhança grande passa a incluir
observações de regiões em que a probabilidade da classe 1 é outra, e aí aparece o viés.

A célula a seguir reproduz a Figura 1 das notas: as fronteiras de decisão do KNN com $k = 1$,
$15$ e $120$, ajustado às 300 observações de treino, com a acurácia de treino e a de teste, nas
10 000 observações novas, no título de cada painel.

In [ ]:
fig, axes = subplots(1, 3, figsize=(11, 3.5))
for ax, k in zip(axes, [1, 15, 120]):
    modelo = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    treino, teste = modelo.score(X_tr, y_tr), modelo.score(X_te, y_te)
    desenhar(ax, modelo, X_tr, y_tr, f"k = {k}\ntreino {treino:.3f}  teste {teste:.3f}")
    print(f"k = {k:3d}: acuracia de treino {treino:.4f}, de teste {teste:.4f}")

print(f"\nas {n} observacoes de treino sao distintas? {len(np.unique(X_tr, axis=0)) == n}")

Os números são os da legenda da Figura 1. Com $k = 1$, a acurácia de treino é exatamente 1:
cada observação de treino é a vizinha mais próxima de si mesma, porque
$d(\mathbf{x}_i, \mathbf{x}_i) = 0$ e as 300 observações são distintas, como a última linha
confere. É o Exercício 2(a) da Lista teórica 10. A fronteira contorna observações isoladas e
forma ilhas em volta delas, e a acurácia de teste, 0,826, é a pior das três. Com $k = 15$ e
$k = 120$, as fronteiras são mais suaves, e as acurácias de teste, 0,888 e 0,889, ficam a menos
de três milésimos dos 0,890 do classificador de Bayes. Que $k = 120$ vá tão bem depende da
população: aqui a fronteira de Bayes é uma reta, e uma fronteira quase reta perde pouco por ser
rígida.

A acurácia de treino cai de 1 para 0,890 e 0,883, enquanto a de teste sobe. Escolher $k$ pela
acurácia de treino levaria a $k = 1$, o pior dos três nas observações novas.

No outro extremo, com $k = n$, os vizinhos de qualquer $\mathbf{x}$ são todas as observações de
treino. Os slides dizem que, nesse caso, $\widehat g$ é constante, igual à classe majoritária.
A célula ajusta o KNN com $k = n = 300$.

In [ ]:
modelo = KNeighborsClassifier(n_neighbors=n).fit(X_tr, y_tr)
print(f"linhas distintas do predict_proba nas observacoes novas: {np.unique(modelo.predict_proba(X_te), axis=0)}")
print(f"classes previstas nas observacoes novas: {np.unique(modelo.predict(X_te))}")
print(f"acuracia de treino {modelo.score(X_tr, y_tr):.4f}, de teste {modelo.score(X_te, y_te):.4f}")

Todas as linhas do `predict_proba` são iguais a $(0{,}5;\,0{,}5)$: as proporções são as da
amostra inteira, 150 observações de cada classe, e não dependem de $\mathbf{x}$. Há um empate em
todo ponto, e o `scikit-learn` fica com a classe que vem primeiro em `classes_`, a classe 0. O
classificador responde 0 para todo $\mathbf{x}$ e acerta metade das observações, no treino e no
teste. Com classes de tamanhos diferentes no treino, como no Exercício 2 da Lista teórica 10,
não há empate, e o classificador constante responde a classe majoritária. É o classificador de
maior viés possível: ele ignora $\mathbf{x}$.

Entre os dois extremos, $k$ é escolhido por validação cruzada (Aula 03), como nos slides.
Usamos 10 dobras estratificadas, que mantêm em cada dobra a proporção das classes, e os valores
ímpares de $k$ de 1 a 199, que evitam empates com duas classes. A figura mostra a acurácia de
validação cruzada de cada $k$ e, para comparação, a acurácia de teste nas 10 000 observações
novas, que na prática não estaria disponível; a linha tracejada é a acurácia do classificador
de Bayes nas mesmas observações.

In [ ]:
grade = np.arange(1, 200, 2)              # os valores impares de k, de 1 a 199
cv = skm.StratifiedKFold(10, shuffle=True, random_state=0)
busca = skm.GridSearchCV(KNeighborsClassifier(), {"n_neighbors": grade}, cv=cv).fit(X_tr, y_tr)
acc_cv = busca.cv_results_["mean_test_score"]
acc_te = np.array([KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr).score(X_te, y_te)
                   for k in grade])

fig, ax = subplots(figsize=(6.2, 3.4))
ax.plot(grade, acc_cv, label="validacao cruzada (10 dobras)")
ax.plot(grade, acc_te, label="teste (10000 observacoes novas)")
ax.axhline(acerto_bayes, ls="--", color="gray", label="classificador de Bayes")
ax.set_xlabel("k"); ax.set_ylabel("acuracia"); ax.legend(fontsize=8)

perto = grade[acc_cv >= acc_cv.max() - 0.01]
print(f"k escolhido: {busca.best_params_['n_neighbors']}, acuracia de validacao cruzada "
      f"{busca.best_score_:.4f}, de teste {busca.score(X_te, y_te):.4f}")
print(f"valores de k a menos de 1 ponto do maximo da validacao cruzada: {len(perto)} de "
      f"{len(grade)}, de {perto.min()} a {perto.max()}")
print(f"maior acuracia de teste da grade: {acc_te.max():.4f}, com k = {grade[acc_te.argmax()]}")

A validação cruzada escolhe $k = 15$, com acurácia estimada de 0,880, e esse classificador
acerta 0,888 das observações novas. As duas curvas são planas numa faixa larga: 71 dos 100
valores da grade ficam a menos de um ponto percentual do máximo da validação cruzada, entre
$k = 7$ e $k = 191$, e o melhor $k$ da grade no teste acerta 0,890, dois milésimos a mais que o
escolhido. Só os valores pequenos de $k$ ficam claramente para trás, e a curva de teste fica
abaixo da linha do classificador de Bayes em toda a grade.

O último slide do KNN adverte que distâncias em alta dimensão podem prejudicar o desempenho, e
as notas dizem por quê: o KNN não seleciona covariáveis, e todas entram na distância euclidiana
com o mesmo peso. Para ver o efeito, a célula a seguir acrescenta às duas covariáveis da
população 2, 5, 10, 20 e 50 covariáveis irrelevantes, independentes da classe, normais com
média 0 e desvio-padrão 0,95, o mesmo das covariáveis relevantes dentro de cada classe. Em cada
caso, mede a acurácia de teste do KNN com $k = 15$ e com o $k$ escolhido de novo pela validação
cruzada.

In [ ]:
rng_ruido = np.random.default_rng(7)
ruido_tr = rng_ruido.normal(0, sigma, size=(n, 50))           # independentes da classe
ruido_te = rng_ruido.normal(0, sigma, size=(len(y_te), 50))

print("irrelevantes    k = 15    k por validacao cruzada")
for extras in [0, 2, 5, 10, 20, 50]:
    Xr_tr = np.hstack([X_tr, ruido_tr[:, :extras]])
    Xr_te = np.hstack([X_te, ruido_te[:, :extras]])
    fixo = KNeighborsClassifier(n_neighbors=15).fit(Xr_tr, y_tr).score(Xr_te, y_te)
    busca_r = skm.GridSearchCV(KNeighborsClassifier(), {"n_neighbors": grade}, cv=cv).fit(Xr_tr, y_tr)
    print(f"     {extras:2d}         {fixo:.4f}    {busca_r.score(Xr_te, y_te):.4f} "
          f"(k = {busca_r.best_params_['n_neighbors']})")

Com 50 covariáveis irrelevantes, a acurácia de teste do KNN com $k = 15$ cai de 0,888 para
0,824. As duas covariáveis relevantes carregam a mesma informação de antes; o que piorou foi a
vizinhança. A distância passou a somar 52 diferenças ao quadrado, 50 delas de ruído, e os
vizinhos de um ponto deixam de ser os pontos parecidos com ele nas covariáveis que importam. A
validação cruzada compensa uma parte da perda escolhendo $k$ maior, 113 com 50 covariáveis
irrelevantes, e chega a 0,8645; ainda assim, fica abaixo dos 0,888 sem elas.

---
## 3. As medidas de impureza

Em regressão, a árvore da Aula 05 particiona o espaço das covariáveis em regiões disjuntas
$R_1, \dots, R_J$ e prevê, em cada região, a média das respostas de treino que caem nela; cada
divisão escolhe a covariável $x_j$ e o corte $t$ que minimizam a soma de quadrados das duas
regiões resultantes. Em classificação, a previsão numa folha $R_k$ passa a ser a moda das
classes das observações de treino que caem nela,
$\widehat g(\mathbf{x}) = \operatorname{moda}\{y_i : \mathbf{x}_i \in R_k\}$ para
$\mathbf{x} \in R_k$, e a soma de quadrados, que não faz sentido com $Y$ qualitativo, dá lugar
a uma medida de impureza das classes. Para uma região $R$, sejam $n_R$ o número de observações
de treino em $R$ e $\hat p_{R,c}$ a proporção da classe $c$ entre elas. As três medidas dos
slides são o erro de classificação, o índice de Gini e a entropia,

$$E(R) = 1 - \max_{c} \hat p_{R,c}, \qquad
  G(R) = \sum_{c \in \mathcal{C}} \hat p_{R,c}\,(1 - \hat p_{R,c}), \qquad
  D(R) = -\sum_{c \in \mathcal{C}} \hat p_{R,c} \log_2 \hat p_{R,c},$$

com a convenção $0 \log_2 0 = 0$. A base 2 do logaritmo é a das notas; outra base só
multiplicaria $D$ por uma constante positiva. Com duas classes e $p = \hat p_{R,1}$, as três
ficam $E = \min(p, 1 - p)$, $G = 2p(1 - p)$ e $D = -p \log_2 p - (1 - p) \log_2 (1 - p)$. A
célula define as três funções de $p$ e as desenha, como a Figura 2 das notas.

In [ ]:
def erro(p):
    return np.minimum(p, 1 - p)


def gini(p):
    return 2 * p * (1 - p)


def entropia(p):
    p = np.asarray(p, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -p * np.log2(p) - (1 - p) * np.log2(1 - p)
    return np.nan_to_num(h)               # a convencao 0 log2 0 = 0


pp = np.linspace(0, 1, 401)
fig, ax = subplots(figsize=(5.2, 3.3))
ax.plot(pp, entropia(pp), label="entropia")
ax.plot(pp, gini(pp), label="Gini")
ax.plot(pp, erro(pp), label="erro de classificacao")
ax.set_xlabel("p (proporcao da classe 1 na regiao)"); ax.set_ylabel("impureza")
ax.legend(fontsize=8)

As três medidas se anulam em $p = 0$ e $p = 1$, quando a região é pura, e são máximas em
$p = 1/2$. A diferença está na forma: o Gini e a entropia são funções estritamente côncavas de
$p$, e o erro de classificação é formado por dois segmentos de reta, um em $[0, 1/2]$ e outro em
$[1/2, 1]$.

Uma divisão de $R$ em $R_1$ e $R_2$, com $n_1$ e $n_2$ observações, é avaliada pela impureza
ponderada $n_1 I(R_1) + n_2 I(R_2)$, em que $I$ é uma das três medidas. Como $n_R I(R)$ não
depende da divisão, a de menor impureza ponderada é a de maior redução de impureza,
$n_R I(R) - [n_1 I(R_1) + n_2 I(R_2)]$. A célula a seguir calcula as três reduções no exemplo
das notas: uma região com 400 observações, 300 da classe 1, dividida em duas de 200, uma com
100 observações da classe 1 ($p_1 = 0{,}5$) e outra só com observações da classe 1
($p_2 = 1$).

In [ ]:
n_R, a_R = 400, 300                  # a regiao: 400 observacoes, 300 da classe 1
filhos = [(200, 100), (200, 200)]    # (observacoes, quantas da classe 1) em cada filho

for nome, medida in [("erro", erro), ("Gini", gini), ("entropia", entropia)]:
    regiao = n_R * medida(a_R / n_R)
    ponderada = sum(n_m * medida(a_m / n_m) for n_m, a_m in filhos)
    print(f"{nome:9s} n_R I(R) = {regiao:8.3f}   n1 I(R1) + n2 I(R2) = {ponderada:8.3f}   "
          f"reducao = {regiao - ponderada:8.3f}")

A divisão é claramente útil: uma das regiões é pura, com 200 observações. O Gini e a entropia
registram isso, com reduções de 50 e 124,5; o erro de classificação registra redução zero. A
conta explica o zero. As proporções da classe 1 são $p = 0{,}75$ na região e $p_1 = 0{,}5$ e
$p_2 = 1$ nas duas partes, as três em $[1/2, 1]$, a metade em que $E = 1 - p$ é uma função
linear de $p$. Como as observações da classe 1 da região são as das duas partes somadas,
$n_R\, p = n_1 p_1 + n_2 p_2$, e a linearidade dá
$n_1 E(R_1) + n_2 E(R_2) = n_1 (1 - p_1) + n_2 (1 - p_2) = n_R (1 - p) = n_R E(R)$. Em
contagens: com duas classes, $n_R E(R)$ é o número de observações de treino de $R$ que a moda
de $R$ classifica errado (Exercício 1(a) da Lista teórica 10), e esse número é 100 na região e
$100 + 0$ nas duas partes.

As notas afirmam que o zero não é coincidência do exemplo. Com duas classes, a redução do erro
de classificação é zero sempre que as duas partes ficam na mesma metade de $[0, 1]$, isto é,
$p_1, p_2 \le 1/2$ ou $p_1, p_2 \ge 1/2$, pelo argumento acima; e o Gini e a entropia, por
serem estritamente côncavos, diminuem sempre que $p_1 \ne p_2$. A célula a seguir confere as
duas afirmações na região do exemplo, percorrendo todas as maneiras de dividi-la em duas partes
não vazias: cada tamanho $n_1$ de 1 a 399 e cada número $a_1$ de observações da classe 1 na
primeira parte compatível com os totais. Os erros são contados como inteiros: numa região com
$n_R$ observações, $a$ delas da classe 1, a moda erra $\min(a, n_R - a)$.

In [ ]:
total = zero_erro = zero_erro_mesma_metade = mesma_metade = 0
gini_nao_cai = entropia_nao_cai = iguais = 0
for n1 in range(1, n_R):
    n2 = n_R - n1
    for a1 in range(max(0, a_R - n2), min(n1, a_R) + 1):   # a1 observacoes da classe 1 na parte 1
        a2 = a_R - a1
        p1, p2 = a1 / n1, a2 / n2
        total += 1
        reducao_erro = min(a_R, n_R - a_R) - (min(a1, n1 - a1) + min(a2, n2 - a2))
        mesma = (p1 <= 0.5 and p2 <= 0.5) or (p1 >= 0.5 and p2 >= 0.5)
        mesma_metade += mesma
        if reducao_erro == 0:
            zero_erro += 1
            zero_erro_mesma_metade += mesma
        gini_nao_cai += n_R * gini(a_R / n_R) - (n1 * gini(p1) + n2 * gini(p2)) <= 1e-9
        entropia_nao_cai += n_R * entropia(a_R / n_R) - (n1 * entropia(p1) + n2 * entropia(p2)) <= 1e-9
        iguais += a1 * n2 == a2 * n1     # p1 = p2, a proporcao da regiao

print(f"divisoes possiveis: {total}")
print(f"reducao do erro igual a zero: {zero_erro}, todas com as partes na mesma metade? "
      f"{zero_erro_mesma_metade == zero_erro}")
print(f"divisoes com as partes na mesma metade de [0, 1]: {mesma_metade}")
print(f"o Gini nao diminui em {gini_nao_cai}, a entropia em {entropia_nao_cai}; "
      f"divisoes com p1 = p2: {iguais}")

São 30 399 divisões possíveis, e em 20 299 delas, dois terços, a redução do erro de
classificação é zero; são exatamente as divisões com as duas partes na mesma metade de
$[0, 1]$. Como a região tem $p = 0{,}75$ e $n_R\,p = n_1 p_1 + n_2 p_2$, as duas partes não
podem ter, ao mesmo tempo, proporção abaixo de $1/2$; estar na mesma metade quer dizer, aqui,
ter as duas proporção de pelo menos $1/2$: as partes mantêm a classe majoritária da região, ou
empatam. O Gini e a
entropia só deixam de diminuir nas 99 divisões em que as duas partes têm a proporção da
própria região, 0,75, e essas divisões de fato não separam nada.

Um algoritmo que crescesse a árvore pelo erro de classificação não distinguiria, nesta região,
dois terços das divisões possíveis de não dividir. Por isso os slides reservam o erro de
classificação para a poda e crescem a árvore pelo Gini ou pela entropia.

---
## 4. Crescer e podar uma árvore de classificação

Os slides terminam com um exemplo: duas meias-luas entrelaçadas, geradas pelo `make_moons`, uma
por classe, com ruído normal de desvio-padrão 0,3 em cada coordenada, e uma árvore ajustada com
`DecisionTreeClassifier(max_depth=None, min_samples_leaf=5)`. Usamos o mesmo número de
observações de treino do exemplo, 350 (70% das 500 do slide), e, como na Seção 2, medimos a
acurácia em 10 000 observações novas da mesma população, em vez de nas 150 de teste do slide.
Com 150 observações, uma acurácia perto de 0,88 tem erro-padrão de cerca de 0,027, maior que as
diferenças entre as árvores que vamos comparar; com 10 000, de cerca de 0,003.

In [ ]:
Xl_tr, yl_tr = make_moons(n_samples=350, noise=0.3, random_state=0)
Xl_te, yl_te = make_moons(n_samples=10_000, noise=0.3, random_state=1)

fig, ax = subplots(figsize=(4.6, 3.8))
desenhar(ax, None, Xl_tr, yl_tr, "as 350 observacoes de treino")
print(f"treino: {len(yl_tr)} observacoes, {np.sum(yl_tr == 1)} da classe 1; "
      f"observacoes novas: {len(yl_te)}")

As duas meias-luas se encaixam uma na outra, e o ruído as mistura perto da fronteira entre
elas, que é curva.

Antes de crescer a árvore inteira, conferimos que o `DecisionTreeClassifier` escolhe cada
divisão como os slides descrevem: entre todas as covariáveis $x_j$ e todos os cortes $t$, a
divisão de $R$ em $R_1 = \{\mathbf{x} \in R : x_j \le t\}$ e
$R_2 = \{\mathbf{x} \in R : x_j > t\}$ de menor impureza ponderada
$n_1 G(R_1) + n_2 G(R_2)$. (O `scikit-learn` põe a igualdade no lado esquerdo, e os slides, no
direito; como os cortes ficam entre valores observados, isso não muda nenhuma divisão.) A
célula ajusta uma árvore de profundidade 1, que faz só a divisão da raiz, e a compara com uma
busca exaustiva, que testa como cortes os pontos médios entre valores consecutivos de cada
covariável. O atributo `tree_` da árvore guarda, para cada nó, o número de observações
(`n_node_samples`) e o Gini (`impurity`).

In [ ]:
toco = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xl_tr, yl_tr)

# busca exaustiva: cada covariavel e cada ponto medio entre valores consecutivos
melhor = (np.inf, None, None)
for j in range(2):
    valores = np.unique(Xl_tr[:, j])
    for t in (valores[:-1] + valores[1:]) / 2:
        esquerda = Xl_tr[:, j] <= t
        ponderada = sum(parte.sum() * gini(yl_tr[parte].mean()) for parte in (esquerda, ~esquerda))
        if ponderada < melhor[0]:
            melhor = (ponderada, j, t)

arv = toco.tree_
print(f"busca exaustiva       : x{melhor[1] + 1} <= {melhor[2]:.4f}, n1 G(R1) + n2 G(R2) = {melhor[0]:.4f}")
print(f"DecisionTreeClassifier: x{arv.feature[0] + 1} <= {arv.threshold[0]:.4f}")
print(f"   observacoes nos nos (raiz, esquerda, direita): {arv.n_node_samples}")
print(f"   Gini dos nos: {arv.impurity.round(4)}")
print(f"   n1 G(R1) + n2 G(R2) = {np.sum(arv.n_node_samples[1:] * arv.impurity[1:]):.4f}")

As duas buscas chegam ao mesmo corte, $x_2 \le 0{,}2295$, com impureza ponderada 110,79. O
`tree_` mostra os números por trás dela: 174 observações e Gini 0,3144 no nó da esquerda, 176 e
0,3186 no da direita, e $174 \times 0{,}3144 + 176 \times 0{,}3186 \approx 110{,}79$. A raiz,
com 175 observações de cada classe, tem Gini $0{,}5$, o máximo com duas classes. A partir
daqui, o `DecisionTreeClassifier` repete a mesma busca em cada nó novo.

Sem restrição, ele divide as regiões até que cada folha seja pura, com observações de uma só
classe. O `min_samples_leaf=5` do exemplo dos slides só admite divisões que deixem pelo menos 5
observações em cada parte, e a árvore para de crescer antes. A célula ajusta as duas árvores e
desenha as fronteiras.

In [ ]:
cheia = DecisionTreeClassifier(random_state=0).fit(Xl_tr, yl_tr)
slides = DecisionTreeClassifier(max_depth=None, min_samples_leaf=5, random_state=0).fit(Xl_tr, yl_tr)

fig, axes = subplots(1, 2, figsize=(9, 3.8))
for ax, (nome, arvore) in zip(axes, [("arvore cheia", cheia), ("min_samples_leaf=5", slides)]):
    treino, teste = arvore.score(Xl_tr, yl_tr), arvore.score(Xl_te, yl_te)
    desenhar(ax, arvore, Xl_tr, yl_tr,
             f"{nome}: {arvore.get_n_leaves()} folhas\ntreino {treino:.3f}  teste {teste:.3f}")
    print(f"{nome:18s}: {arvore.get_n_leaves()} folhas, profundidade {arvore.get_depth()}, "
          f"acuracia de treino {treino:.4f}, de teste {teste:.4f}")

A árvore cheia tem 44 folhas e acerta todas as observações de treino: cada folha é pura, e cada
observação de treino cai numa folha da sua classe. Nas observações novas, acerta 0,871. Na
figura, ela recorta pequenos retângulos em volta de observações isoladas, como as ilhas do KNN
com $k = 1$. A árvore com `min_samples_leaf=5` tem 24 folhas, acerta 0,940 do treino e 0,895 das
observações novas. As duas fronteiras são feitas de segmentos paralelos aos eixos, porque cada
divisão usa uma covariável só.

Os slides admitem as duas medidas para crescer: com a entropia, troca-se $G$ por $D$ na
impureza ponderada. No `DecisionTreeClassifier`, isso é o argumento `criterion="entropy"`;
segundo a documentação, o terceiro valor aceito, `"log_loss"`, é a mesma entropia com outro
nome. A célula cresce as mesmas duas árvores com a entropia, compara-as com as do Gini e
imprime a impureza da raiz, que mostra em que base o `scikit-learn` calcula a entropia.

In [ ]:
for nome, folha_min, arv_gini in [("cheia", 1, cheia), ("min_samples_leaf=5", 5, slides)]:
    arv_ent = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=folha_min,
                                     random_state=0).fit(Xl_tr, yl_tr)
    print(f"arvore {nome}:")
    for crit, arvore in [("Gini", arv_gini), ("entropia", arv_ent)]:
        raiz = arvore.tree_
        print(f"   {crit:8s}: raiz x{raiz.feature[0] + 1} <= {raiz.threshold[0]:.3f} "
              f"(impureza {raiz.impurity[0]:.4f}), {arvore.get_n_leaves()} folhas, "
              f"acuracia de teste {arvore.score(Xl_te, yl_te):.4f}")
    iguais = np.mean(arv_gini.predict(Xl_te) == arv_ent.predict(Xl_te))
    print(f"   as duas classificam igual {iguais:.2%} das observacoes novas")

A entropia da raiz é 1, o máximo em base 2 com duas classes equilibradas: o `scikit-learn`
calcula a entropia em base 2, como as notas. As duas medidas escolhem o mesmo corte na raiz,
$x_2 \le 0{,}230$. Com `min_samples_leaf=5`, as árvores têm 24 e 22 folhas, acertam 0,895 e
0,891 das observações novas e classificam igual 97,8% delas; as árvores cheias concordam em
93,6% das observações novas e acertam 0,871 e 0,860. As diferenças de acurácia, de 0,4 e 1,1
ponto, vêm de uma amostra de treino só; a última célula desta seção mede se elas se repetem em
outras amostras.

A árvore cheia acerta todo o treino e erra mais nas observações novas: como em regressão, ela é
grande demais. A segunda etapa dos slides é a poda por custo–complexidade, com $\alpha$
escolhido por validação cruzada. Na Aula 05, para cada $\alpha \ge 0$, a poda escolhia a
subárvore $T$ da árvore cheia que minimiza a soma de quadrados das folhas mais
$\alpha\,|T|$, em que $|T|$ é o número de folhas, e os slides desta aula dizem que, em
classificação, o erro de classificação é o usado na poda.

O `scikit-learn` faz a poda em duas etapas. Na primeira, o argumento `ccp_alpha` escolhe a
subárvore que minimiza

$$\sum_{m=1}^{|T|} \frac{n_m}{n}\, I(R_m) + \alpha\,|T|,$$

em que $R_1, \dots, R_{|T|}$ são as folhas de $T$, $n_m$ é o número de observações de treino em
$R_m$ e $I$ é a medida usada para crescer, aqui o Gini, e não o erro de classificação. O método
`cost_complexity_pruning_path` devolve os valores de $\alpha$ em que essa subárvore muda
(`ccp_alphas`) e, para cada um, a soma ponderada das impurezas das folhas (`impurities`). Na
segunda etapa, a validação cruzada com `scoring="accuracy"` escolhe $\alpha$ entre esses
valores pela acurácia nas observações deixadas de fora, isto é, pelo complemento da taxa de
erro: é aqui que o erro de classificação entra. É o que o [AME] §8.3 descreve, ao dizer que na
poda se usa a proporção de erros num conjunto de validação como estimativa do risco, e o que o
laboratório do [ISLP] §8.3.1 faz. A célula calcula a sequência, escolhe $\alpha$ com 10 dobras
estratificadas e confere, na subárvore escolhida, o que o `impurities` mede.

In [ ]:
caminho = cheia.cost_complexity_pruning_path(Xl_tr, yl_tr)
alphas = caminho.ccp_alphas[:-1]          # o ultimo alpha poda a arvore ate a raiz
busca = skm.GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas},
                         cv=cv, scoring="accuracy").fit(Xl_tr, yl_tr)
podada = busca.best_estimator_
alpha = busca.best_params_["ccp_alpha"]
print(f"alpha escolhido: {alpha:.4f}, {podada.get_n_leaves()} folhas, acuracia de validacao "
      f"cruzada {busca.best_score_:.4f}, de teste {podada.score(Xl_te, yl_te):.4f}")

# o que o caminho chama de impurities, na subarvore escolhida
arv = podada.tree_
folha = arv.children_left == -1           # as folhas nao tem filhos
soma_gini = np.sum(arv.n_node_samples[folha] * arv.impurity[folha]) / arv.n_node_samples[0]
i = np.flatnonzero(alphas == alpha)[0]
print(f"\nimpurities do caminho nesse alpha     : {caminho.impurities[i]:.4f}")
print(f"soma dos Gini das folhas, pesos n_m/n : {soma_gini:.4f}")
print(f"taxa de erro de treino da subarvore   : {1 - podada.score(Xl_tr, yl_tr):.4f}")

A validação cruzada escolhe $\alpha = 0{,}0125$, uma subárvore de 4 folhas, com acurácia
estimada de 0,871; nas observações novas, ela acerta 0,880, mais que a árvore cheia (0,871) e
menos que a de `min_samples_leaf=5` (0,895). As três últimas linhas confirmam o que a poda do
`scikit-learn` usa: o `impurities` do caminho, 0,1867, é a soma dos Gini das 4 folhas
ponderados por $n_m/n$, e não a taxa de erro de treino da mesma subárvore, 0,1057.

A figura a seguir acompanha três acurácias ao longo da sequência de subárvores, como a Figura
8.5 do [ISLP] faz em regressão: a de treino, a de validação cruzada, com uma barra de um
erro-padrão (o desvio-padrão entre as dez dobras dividido por $\sqrt{10}$), e a de teste.

In [ ]:
folhas, treino, teste = [], [], []
for a in alphas:
    arvore = DecisionTreeClassifier(random_state=0, ccp_alpha=a).fit(Xl_tr, yl_tr)
    folhas.append(arvore.get_n_leaves())
    treino.append(arvore.score(Xl_tr, yl_tr))
    teste.append(arvore.score(Xl_te, yl_te))
acc_cv = busca.cv_results_["mean_test_score"]
ep_cv = busca.cv_results_["std_test_score"] / np.sqrt(10)

fig, ax = subplots(figsize=(6.2, 3.6))
ax.plot(folhas, treino, "o-", ms=3, label="treino")
ax.errorbar(folhas, acc_cv, yerr=ep_cv, fmt="o-", ms=3, capsize=2, label="validacao cruzada")
ax.plot(folhas, teste, "o-", ms=3, label="teste")
ax.set_xlabel("numero de folhas"); ax.set_ylabel("acuracia"); ax.legend(fontsize=8)

entre = (np.array(folhas) >= 4) & (np.array(folhas) <= 12)
print(f"validacao cruzada entre 4 e 12 folhas: de {acc_cv[entre].min():.4f} a "
      f"{acc_cv[entre].max():.4f}; erro-padrao de {ep_cv[entre].min():.4f} a {ep_cv[entre].max():.4f}")
print(f"maior acuracia de teste: {max(teste):.4f}, com {folhas[int(np.argmax(teste))]} folhas")

A acurácia de treino nunca diminui quando o número de folhas aumenta: ao dividir uma região, a
moda de cada parte acerta, entre as observações dessa parte, pelo menos tantas quanto a moda da
região inteira acertava, e a árvore cheia, de folhas puras, chega a 1. A de validação cruzada é quase plana entre 4 e 12
folhas, com valores de 0,857 a 0,871, e as diferenças entre essas subárvores são menores que o
erro-padrão de cada uma, de 0,015 a 0,018. Com essa dispersão, a validação cruzada não tem como
separar essas subárvores, e o máximo cair em 4 folhas é em boa parte acaso da amostra; no
teste, o máximo fica em 10 folhas, com 0,899. É o mesmo quadro da Figura 8.5 do [ISLP], em que a
validação cruzada escolhe 3 folhas e o erro de teste é mínimo em 10.

Com uma amostra de treino só, não dá para comparar as três maneiras de controlar o tamanho da
árvore: com outra amostra, a validação cruzada escolheria outro $\alpha$, e a árvore com
`min_samples_leaf=5` seria outra. A célula a seguir repete os ajustes em 50 amostras de treino
de 350 observações da mesma população e mede, em cada uma, a acurácia nas mesmas 10 000
observações novas da árvore cheia, da árvore com `min_samples_leaf=5`, com o Gini e com a
entropia, e da árvore podada por validação cruzada. Leva alguns segundos.

In [ ]:
linhas = []
for r in range(50):
    X_r, y_r = make_moons(n_samples=350, noise=0.3, random_state=100 + r)
    cheia_r = DecisionTreeClassifier(random_state=0).fit(X_r, y_r)
    alphas_r = cheia_r.cost_complexity_pruning_path(X_r, y_r).ccp_alphas[:-1]
    busca_r = skm.GridSearchCV(DecisionTreeClassifier(random_state=0), {"ccp_alpha": alphas_r},
                               cv=cv, scoring="accuracy").fit(X_r, y_r)
    gini_r = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(X_r, y_r)
    ent_r = DecisionTreeClassifier(criterion="entropy", min_samples_leaf=5,
                                   random_state=0).fit(X_r, y_r)
    linhas.append({"cheia": cheia_r.score(Xl_te, yl_te),
                   "min_samples_leaf=5": gini_r.score(Xl_te, yl_te),
                   "podada": busca_r.score(Xl_te, yl_te),
                   "entropia, min_samples_leaf=5": ent_r.score(Xl_te, yl_te),
                   "folhas da podada": busca_r.best_estimator_.get_n_leaves(),
                   "Gini e entropia iguais": np.mean(gini_r.predict(Xl_te) == ent_r.predict(Xl_te))})
tab = pd.DataFrame(linhas)

print(tab.describe().loc[["mean", "std", "min", "max"]].round(3).T.to_string())
print(f"\na podada acerta mais que a cheia em {np.sum(tab['podada'] > tab['cheia'])} das 50 amostras")
print(f"a de min_samples_leaf=5 acerta mais que a cheia em {np.sum(tab['min_samples_leaf=5'] > tab['cheia'])}")
print(f"a podada acerta mais que a de min_samples_leaf=5 em "
      f"{np.sum(tab['podada'] > tab['min_samples_leaf=5'])}")
dif = tab["min_samples_leaf=5"] - tab["entropia, min_samples_leaf=5"]
print(f"com min_samples_leaf=5, o Gini acerta mais que a entropia em {np.sum(dif > 0)}; "
      f"diferenca media {dif.mean():.4f}")

Em média, a árvore cheia acerta 0,867 das observações novas; a de `min_samples_leaf=5`, 0,886; a
podada por validação cruzada, 0,885. Limitar o tamanho das folhas ou podar melhora a árvore
cheia em cerca de dois pontos, e isso acontece em quase todas as amostras: em 47 das 50 com
`min_samples_leaf=5` e em 45 com a poda. Entre essas duas, não há vencedora: a podada acerta mais
em 25 das 50. A poda tem a vantagem de escolher o tamanho pelos dados, e o preço é a
variabilidade dessa escolha: o número de folhas da podada vai de 3 a 47 entre as amostras.

Entre o Gini e a entropia, a diferença média é de 0,3 ponto, pequena diante da variação de uma
amostra para outra, de cerca de um ponto; o Gini acerta mais em 31 das 50 amostras, e as duas
árvores classificam igual, em média, 96% das observações novas. É o que as notas dizem: a
escolha entre as duas medidas raramente muda o resultado, e vale mais gastar a validação
cruzada no tamanho da árvore.

---
## 5. *Bagging* e florestas aleatórias

Os slides lembram que as árvores de classificação podem ser agregadas em *bagging* e em
florestas aleatórias, como as de regressão na Aula 05: cada uma das $B$ árvores é ajustada a
uma amostra *bootstrap* do treino, e, na floresta, cada divisão só pode usar $m$ das $p$
covariáveis, sorteadas a cada nó. A regra das notas para classificação é $m \approx \sqrt{p}$;
com $p = 2$, o `max_features="sqrt"` padrão do `RandomForestClassifier` dá $m = 1$, e em cada nó
a árvore sorteia uma das duas covariáveis e só pode cortar por ela.

Em regressão, as $B$ previsões eram agregadas pela média. Em classificação, o [AME] §8.4 e o
[ISLP] §8.2.1 agregam pelo voto: cada árvore vota numa classe, e vence a mais votada. O
`scikit-learn` agrega de outro modo: tira a média das $B$ estimativas de
$P(Y = c \mid \mathbf{X} = \mathbf{x})$, as proporções das classes nas folhas em que
$\mathbf{x}$ cai, e escolhe a classe de maior média. A célula ajusta o *bagging* e a floresta,
com $B = 300$ árvores cheias, e os compara com as duas árvores da Seção 4.

In [ ]:
bagging = BaggingClassifier(DecisionTreeClassifier(random_state=0), n_estimators=300,
                            random_state=0).fit(Xl_tr, yl_tr)
floresta = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xl_tr, yl_tr)
print(f"covariaveis sorteadas em cada no da floresta: {floresta.estimators_[0].max_features_}\n")

fig, axes = subplots(1, 2, figsize=(9, 3.8))
for ax, (nome, m) in zip(axes, [("bagging (B = 300)", bagging), ("floresta (B = 300)", floresta)]):
    desenhar(ax, m, Xl_tr, yl_tr,
             f"{nome}\ntreino {m.score(Xl_tr, yl_tr):.3f}  teste {m.score(Xl_te, yl_te):.3f}")
for nome, m in [("arvore cheia", cheia), ("min_samples_leaf=5", slides),
                ("bagging", bagging), ("floresta", floresta)]:
    print(f"{nome:18s}: acuracia de treino {m.score(Xl_tr, yl_tr):.4f}, de teste {m.score(Xl_te, yl_te):.4f}")

O *bagging* e a floresta acertam 0,893 e 0,895 das observações novas, contra 0,871 da árvore
cheia, e ficam no nível da árvore com `min_samples_leaf=5`, 0,895. Como a árvore cheia, os dois
acertam todas as observações de treino, e mesmo assim erram menos nas novas: a acurácia de
treino igual a 1 não separa um ajuste que generaliza de um que não generaliza, como já se viu com
o KNN de $k = 1$. Na figura, as duas fronteiras continuam feitas de segmentos paralelos aos
eixos, mas em degraus bem menores que os de uma árvore só.

Com árvores cheias, o voto e a média das probabilidades dão sempre a mesma classe. Cada folha de
uma árvore cheia é pura, e a árvore estima $P(Y = 1 \mid \mathbf{X} = \mathbf{x})$ por 0 ou por
1; a média das $B$ estimativas é, então, a proporção de árvores que votam na classe 1, e a
classe de maior média é a mais votada. Com folhas impuras, isso deixa de valer. A célula a
seguir compara as duas regras na floresta de árvores cheias e numa floresta com
`min_samples_leaf=5`, em que as folhas guardam proporções intermediárias.

In [ ]:
floresta5 = RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                   random_state=0).fit(Xl_tr, yl_tr)

for nome, m in [("floresta de arvores cheias", floresta), ("floresta com min_samples_leaf=5", floresta5)]:
    votos = np.mean([arvore.predict(Xl_te) for arvore in m.estimators_], axis=0)  # proporcao de votos na classe 1
    media = m.predict_proba(Xl_te)[:, 1]                                          # media das probabilidades
    pelo_voto = (votos > 0.5).astype(int)
    print(f"{nome}:")
    print(f"   valores distintos da estimativa da primeira arvore: "
          f"{np.unique(m.estimators_[0].predict_proba(Xl_te)[:, 1]).size}")
    print(f"   maior diferenca entre a proporcao de votos e a media: {np.abs(votos - media).max():.4f}")
    print(f"   observacoes novas em que as duas regras discordam: {np.sum(pelo_voto != m.predict(Xl_te))}")
    print(f"   acuracia pelo voto {np.mean(pelo_voto == yl_te):.4f}, pela media {m.score(Xl_te, yl_te):.4f}")

Na floresta de árvores cheias, a primeira árvore estima a probabilidade só com os valores 0 e
1, como toda árvore de folhas puras, a proporção de votos coincide com a média das
probabilidades em todas as 10 000 observações novas, e as duas regras dão as mesmas classes. Na floresta com `min_samples_leaf=5`, a primeira
árvore usa 12 valores diferentes, a proporção de votos e a média chegam a diferir em 0,13, e as
duas regras discordam em 104 das 10 000 observações novas; a média acerta 0,902, e o voto,
0,899. A média leva em conta o quanto cada árvore está segura da sua classe, e o voto, só a
classe; nesta amostra, a diferença entre as duas regras foi de três décimos de ponto.

---
## Resumo

| Conceito | Onde apareceu | O que vimos |
|---|---|---|
| o voto e as proporções | §2 | o `predict` é a classe mais votada entre os $k$ vizinhos, e o `predict_proba`, a proporção de cada classe; no ponto médio entre os centros, o desvio-padrão da proporção cai de 0,50 ($k = 1$) para 0,036 ($k = 120$) |
| os extremos de $k$ | §2 | $k = 1$ acerta todo o treino e é o pior no teste (0,826); $k = n$ responde a mesma classe em todo ponto |
| $k$ por validação cruzada | §2 | a validação cruzada escolhe $k = 15$, que acerta 0,888, perto dos 0,890 do classificador de Bayes |
| covariáveis irrelevantes | §2 | 50 delas derrubam o KNN com $k = 15$ de 0,888 para 0,824 |
| as três impurezas | §3 | na região de 400 das notas, o erro de classificação não registra uma parte pura; em dois terços das divisões possíveis dela, a redução do erro é zero |
| crescer | §4 | o `scikit-learn` escolhe a divisão de menor $n_1 G(R_1) + n_2 G(R_2)$; a árvore cheia acerta todo o treino e 0,871 no teste |
| podar | §4 | o `ccp_alpha` poda pela impureza das folhas, e a validação cruzada escolhe $\alpha$ pela acurácia; em 50 amostras, podar ou limitar as folhas melhora a árvore cheia em cerca de dois pontos |
| Gini e entropia | §4 | mesmo corte na raiz e, em média, 96% das previsões iguais; diferença média de 0,3 ponto |
| *bagging* e floresta | §5 | 0,893 e 0,895 no teste, contra 0,871 da árvore cheia; com árvores cheias, o voto e a média das probabilidades coincidem |

**Leitura recomendada.** [AME] §8.6 (o KNN em classificação), §8.3 (árvores de classificação; a
Observação 8.2 e o parágrafo seguinte explicam por que se cresce pelo Gini e dizem que a poda usa
a proporção de erros num conjunto de validação) e §8.4 (*bagging* e florestas, agregados pela
moda). [ISLP] §2.2.3 (o KNN como classificador, comparado ao classificador de Bayes nas Figuras
2.14 a 2.17), §8.1.2 (árvores de classificação, o Gini, a entropia e a recomendação do erro de
classificação na poda), §8.2.1 e §8.2.2 (*bagging* e florestas, com o voto da maioria) e os
laboratórios §4.7.6 (KNN) e §8.3.1 (árvore de classificação, com a poda por `ccp_alpha` escolhido
por validação cruzada).

**A seguir.** A Aula 10 encerra a parte de classificação do curso. As aulas extras tratam de
outros problemas: a E1 e a E2, de dados sem resposta, com o agrupamento e a redução de
dimensionalidade, e a E3 aplica classificadores a textos.